[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/08_rmsnorm.ipynb)

# 🟡 Medium: Implement RMSNorm

Implement **Root Mean Square Layer Normalization** — the normalization used in LLaMA, Gemma, etc.

$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot w, \quad \text{RMS}(x) = \sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}$$

### Signature
```python
def rms_norm(x: torch.Tensor, weight: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    # Normalize over the last dimension. No mean subtraction (unlike LayerNorm).
```

### Rules
- Do **NOT** use any built-in norm layers
- Normalize over `dim=-1`
- Must support autograd

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

# RMSNorm 是 LayerNorm 的简化版，去掉了减均值的步骤，只做缩放。
# LayerNorm: x_norm = (x - mean) / sqrt(var + eps)，有 gamma 和 beta 两个参数
# RMSNorm:   x_norm = x / rms，只有 weight（相当于 gamma），没有 beta
#
# 为什么去掉均值也行？
# 论文认为 LayerNorm 的主要效果来自缩放而非平移，去掉均值影响很小，
# 但省掉了计算量，在大模型里速度收益明显。
#
# 哪些模型用 RMSNorm？LLaMA、Mistral、Gemma、Qwen 等现代主流 LLM。
# GPT-2 用的是 LayerNorm，之后的大模型大多迁移到了 RMSNorm。
#
# 为什么 LLM 不用 BatchNorm？
# BatchNorm 对 batch 内同一特征做归一化，依赖其他样本的统计量。
# 推理时 batch=1，统计量不稳定；序列长度不同时 padding 也会污染统计。
# LayerNorm/RMSNorm 只对单个样本自身归一化，不依赖 batch，更适合 NLP。

def rms_norm(x, weight, eps=1e-6):
    x_squared = x.pow(2)
    x_squared_mean = x_squared.mean(dim=-1, keepdim=True)
    rms = torch.sqrt(x_squared_mean + eps)
    return weight * x / rms


In [4]:
# 🧪 Debug
x = torch.randn(2, 8)
w = torch.ones(8)
out = rms_norm(x, w)
print("Output shape:", out.shape)
print("RMS of output:", out.pow(2).mean(dim=-1).sqrt())  # should be ~1

Output shape: torch.Size([2, 8])
RMS of output: tensor([1.0000, 1.0000])


In [5]:
from torch_judge import check
check('rmsnorm')


🧪 Testing: Implement RMSNorm (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Basic behavior (0.6ms)
  ✅ [2/4] With learned weight (0.1ms)
  ✅ [3/4] 3-D input (0.1ms)
  ✅ [4/4] Gradient flow (0.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (1.3ms total)
  Progress saved. Run status() to see your dashboard.

